# 99 · Comparación final

Ordena por accuracy y mantiene visible origen y protocolo.

Incluye los diez intentos experimentales en `results/experiment_attempts.csv`; el holdout agrupado se reutilizó para selección.

In [ ]:
from pathlib import Path
import sys
ROOT=Path.cwd().resolve()
for p in (ROOT,*ROOT.parents):
    if (p/'src'/'ap2026q2_common.py').exists(): ROOT=p; break
sys.path.insert(0,str(ROOT/'src'))
from ap2026q2_common import *
print('Repo:',ROOT,'Datos:',project_data_dir())

In [ ]:
hist=historical_scores().copy(); hist['source']='histórico'; hist=hist.rename(columns={'model':'model_name'})
exp_path=ROOT/'results'/'experiment_attempts.csv'
exp=pd.read_csv(exp_path) if exp_path.exists() else pd.DataFrame()
if not exp.empty:
    exp['source']='ciclo de 10 intentos'; exp=exp.rename(columns={'model':'model_name','protocol':'split'})
    combined=pd.concat([hist,exp],ignore_index=True,sort=False)
else: combined=hist
local_path=processed_dir()/'experiment_runs.csv'
if local_path.exists():
    local=pd.read_csv(local_path);local['source']='run local';local['model_name']=local['model'];local['split']=local.get('split','stratified_random_80_20')
    combined=pd.concat([combined,local],ignore_index=True,sort=False)
display(combined.sort_values('accuracy',ascending=False)[['model_name','source','split','accuracy','balanced_accuracy','macro_f1']].reset_index(drop=True))
best=combined.sort_values('accuracy',ascending=False).iloc[0]
print('Meta:',.94,'| Mejor accuracy:',round(float(best.accuracy),4),'| Modelo:',best.model_name)

In [ ]:
key='model_key' if 'model_key' in scores else 'model'
best=scores.sort_values('accuracy',ascending=False).drop_duplicates(key)
display(best[['model_name','source','protocol','accuracy','balanced_accuracy','macro_f1']].sort_values('accuracy',ascending=False).reset_index(drop=True))
print('Meta:',.94,'| Mejor accuracy:',round(float(best.accuracy.max()),4))